# Broker Behavior Classification Pipeline

## Objective
Build a three-stage classification pipeline to predict broker classes for the following month:

### Stage 1: Others vs nonOthers (Target: 90% accuracy)
- **Others (O)**: < 2 competitive loans next month
- **nonOthers**: ≥ 2 competitive loans next month

### Stage 2: Confirm nonOthers (Target: 80% accuracy)
- Take predicted nonOthers from Stage 1
- Confirm they remain nonOthers next month

### Stage 3: Whale/Dolphin/Minnow Classification (Target: 70% accuracy)
- **Whale (W)**: ≥ 6 competitive loans
- **Dolphin (D)**: 4-5 competitive loans
- **Minnow (M)**: 2-3 competitive loans

## Data
- Training data: 6 months of historical broker behavior
- Features: UWM metrics, Google Reviews, competitor loans, behavioral flags

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')

print("Libraries imported successfully!")

## 1. Load and Explore Data

In [ ]:
# Load the data
# NOTE: Update this path if your file is in a different location
df = pd.read_excel('../data/raw/Master_One_Sheet.xlsx')

print(f"Data shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Basic data info
print("Data Info:")
df.info()

print("\n" + "="*50)
print("Missing Values:")
print(df.isnull().sum())

print("\n" + "="*50)
print("Unique months:")
print(df['Month'].unique())

print("\n" + "="*50)
print("Current month class distribution:")
print(df['Actual Class THIS Month'].value_counts())

In [ ]:
# Explore the target variable (Competitor Loans)
print("Competitor Loans Statistics:")
print(df['Competitor Loans this Month'].describe())

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
df['Competitor Loans this Month'].hist(bins=30, edgecolor='black')
plt.xlabel('Competitor Loans')
plt.ylabel('Frequency')
plt.title('Distribution of Competitor Loans')

plt.subplot(1, 2, 2)
df['Actual Class THIS Month'].value_counts().plot(kind='bar', edgecolor='black')
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Current Month Class Distribution')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

## 2. Feature Engineering and Target Creation

In [ ]:
# Create a copy for processing
data = df.copy()

# Sort by Account Name and Month to ensure proper ordering
data = data.sort_values(['Account Name', 'Month']).reset_index(drop=True)

# Create target variables for NEXT month
# We need to shift the competitor loans backward to get "next month's" value
data['Next_Month_Competitor_Loans'] = data.groupby('Account Name')['Competitor Loans this Month'].shift(-1)

# Create classification targets
def classify_stage1(cl):
    """Stage 1: Others vs nonOthers"""
    if pd.isna(cl):
        return np.nan
    return 'nonOthers' if cl >= 2 else 'Others'

def classify_stage3(cl):
    """Stage 3: Whale/Dolphin/Minnow (only for nonOthers)"""
    if pd.isna(cl) or cl < 2:
        return np.nan
    if cl >= 6:
        return 'Whale'
    elif cl >= 4:
        return 'Dolphin'
    else:
        return 'Minnow'

# Create target columns
data['Target_Stage1'] = data['Next_Month_Competitor_Loans'].apply(classify_stage1)
data['Target_Stage3'] = data['Next_Month_Competitor_Loans'].apply(classify_stage3)

print("Target variables created!")
print(f"\nStage 1 Target Distribution:")
print(data['Target_Stage1'].value_counts())
print(f"\nStage 3 Target Distribution (nonOthers only):")
print(data['Target_Stage3'].value_counts())

In [ ]:
# Define feature columns
# Exclude identifier columns and target-related columns
exclude_cols = ['Month', 'Account Name', 'Owner Name', '2 Records ONLY', 
                'Actual Class THIS Month', 'Actual Next Month (N->O)', 
                'PBA Score (Prediction)', 'Predicted Class NEXT Month',
                'Next_Month_Competitor_Loans', 'Target_Stage1', 'Target_Stage3']

feature_cols = [col for col in data.columns if col not in exclude_cols]

print(f"Number of features: {len(feature_cols)}")
print(f"\nFeature columns:")
for i, col in enumerate(feature_cols, 1):
    print(f"{i}. {col}")

In [ ]:
# Handle missing values in features
print("Missing values in features before cleaning:")
print(data[feature_cols].isnull().sum()[data[feature_cols].isnull().sum() > 0])

# Fill missing values with 0 for numeric columns (assuming 0 is meaningful)
# Adjust this based on your domain knowledge
data[feature_cols] = data[feature_cols].fillna(0)

print("\nMissing values after cleaning:")
print(data[feature_cols].isnull().sum().sum())

## 3. Stage 1: Others vs nonOthers Classifier
**Target Accuracy: 90%**

In [ ]:
# Prepare data for Stage 1
# Remove rows where we don't have next month data (last month for each broker)
stage1_data = data[data['Target_Stage1'].notna()].copy()

X_stage1 = stage1_data[feature_cols]
y_stage1 = stage1_data['Target_Stage1']

print(f"Stage 1 dataset shape: {X_stage1.shape}")
print(f"\nClass distribution:")
print(y_stage1.value_counts())
print(f"\nClass proportions:")
print(y_stage1.value_counts(normalize=True))

In [ ]:
# Split data for Stage 1
X_train_s1, X_test_s1, y_train_s1, y_test_s1 = train_test_split(
    X_stage1, y_stage1, test_size=0.2, random_state=42, stratify=y_stage1
)

print(f"Training set size: {X_train_s1.shape}")
print(f"Test set size: {X_test_s1.shape}")
print(f"\nTraining set class distribution:")
print(y_train_s1.value_counts())

In [ ]:
# Test multiple models for Stage 1
models_s1 = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results_s1 = {}

print("Training and evaluating Stage 1 models...\n")
print("="*60)

for name, model in models_s1.items():
    # Train
    model.fit(X_train_s1, y_train_s1)
    
    # Predict
    y_pred = model.predict(X_test_s1)
    
    # Evaluate
    accuracy = accuracy_score(y_test_s1, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_s1, y_train_s1, cv=5, scoring='accuracy')
    
    results_s1[name] = {
        'model': model,
        'accuracy': accuracy,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    print(f"{name}:")
    print(f"  Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
    print(f"\nClassification Report:")
    print(classification_report(y_test_s1, y_pred))
    print("="*60)

# Select best model
best_model_s1 = max(results_s1.items(), key=lambda x: x[1]['accuracy'])
print(f"\n🏆 Best Stage 1 Model: {best_model_s1[0]} with {best_model_s1[1]['accuracy']*100:.2f}% accuracy")
print(f"Target: 90% accuracy - {'✓ ACHIEVED!' if best_model_s1[1]['accuracy'] >= 0.90 else '❌ Not yet achieved'}")

In [ ]:
# Confusion matrix for best Stage 1 model
best_s1_name = best_model_s1[0]
best_s1_model = best_model_s1[1]['model']
y_pred_s1 = best_s1_model.predict(X_test_s1)

cm = confusion_matrix(y_test_s1, y_pred_s1)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['nonOthers', 'Others'],
            yticklabels=['nonOthers', 'Others'])
plt.title(f'Stage 1 Confusion Matrix - {best_s1_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

In [ ]:
# Feature importance for Stage 1 (if using tree-based model)
if best_s1_name in ['Random Forest', 'Gradient Boosting']:
    feature_importance_s1 = pd.DataFrame({
        'feature': feature_cols,
        'importance': best_s1_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print("Top 15 Most Important Features for Stage 1:")
    print(feature_importance_s1.head(15))
    
    plt.figure(figsize=(10, 8))
    plt.barh(feature_importance_s1.head(15)['feature'], 
             feature_importance_s1.head(15)['importance'])
    plt.xlabel('Importance')
    plt.title('Top 15 Features - Stage 1')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 4. Stage 2: Confirm nonOthers Classifier
**Target Accuracy: 80%**

Take brokers predicted as nonOthers in Stage 1 and confirm they stay nonOthers.

In [ ]:
# Stage 2: Filter for brokers who are currently nonOthers
# (In real pipeline, these would be Stage 1 predictions)
stage2_data = stage1_data[stage1_data['Competitor Loans this Month'] >= 2].copy()

X_stage2 = stage2_data[feature_cols]
y_stage2 = stage2_data['Target_Stage1']

print(f"Stage 2 dataset shape: {X_stage2.shape}")
print(f"\nClass distribution:")
print(y_stage2.value_counts())
print(f"\nClass proportions:")
print(y_stage2.value_counts(normalize=True))

In [ ]:
# Split data for Stage 2
X_train_s2, X_test_s2, y_train_s2, y_test_s2 = train_test_split(
    X_stage2, y_stage2, test_size=0.2, random_state=42, stratify=y_stage2
)

print(f"Training set size: {X_train_s2.shape}")
print(f"Test set size: {X_test_s2.shape}")

In [ ]:
# Test multiple models for Stage 2
models_s2 = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results_s2 = {}

print("Training and evaluating Stage 2 models...\n")
print("="*60)

for name, model in models_s2.items():
    # Train
    model.fit(X_train_s2, y_train_s2)
    
    # Predict
    y_pred = model.predict(X_test_s2)
    
    # Evaluate
    accuracy = accuracy_score(y_test_s2, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_s2, y_train_s2, cv=5, scoring='accuracy')
    
    results_s2[name] = {
        'model': model,
        'accuracy': accuracy,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    print(f"{name}:")
    print(f"  Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
    print(f"\nClassification Report:")
    print(classification_report(y_test_s2, y_pred))
    print("="*60)

# Select best model
best_model_s2 = max(results_s2.items(), key=lambda x: x[1]['accuracy'])
print(f"\n🏆 Best Stage 2 Model: {best_model_s2[0]} with {best_model_s2[1]['accuracy']*100:.2f}% accuracy")
print(f"Target: 80% accuracy - {'✓ ACHIEVED!' if best_model_s2[1]['accuracy'] >= 0.80 else '❌ Not yet achieved'}")

## 5. Stage 3: Whale/Dolphin/Minnow Classifier
**Target Accuracy: 70%**

Classify confirmed nonOthers into three classes based on competitive loan volume.

In [ ]:
# Stage 3: Filter for confirmed nonOthers only
stage3_data = stage1_data[stage1_data['Target_Stage3'].notna()].copy()

X_stage3 = stage3_data[feature_cols]
y_stage3 = stage3_data['Target_Stage3']

print(f"Stage 3 dataset shape: {X_stage3.shape}")
print(f"\nClass distribution:")
print(y_stage3.value_counts())
print(f"\nClass proportions:")
print(y_stage3.value_counts(normalize=True))

In [ ]:
# Split data for Stage 3
X_train_s3, X_test_s3, y_train_s3, y_test_s3 = train_test_split(
    X_stage3, y_stage3, test_size=0.2, random_state=42, stratify=y_stage3
)

print(f"Training set size: {X_train_s3.shape}")
print(f"Test set size: {X_test_s3.shape}")

In [ ]:
# Test multiple models for Stage 3
models_s3 = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42, multi_class='multinomial'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

results_s3 = {}

print("Training and evaluating Stage 3 models...\n")
print("="*60)

for name, model in models_s3.items():
    # Train
    model.fit(X_train_s3, y_train_s3)
    
    # Predict
    y_pred = model.predict(X_test_s3)
    
    # Evaluate
    accuracy = accuracy_score(y_test_s3, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train_s3, y_train_s3, cv=5, scoring='accuracy')
    
    results_s3[name] = {
        'model': model,
        'accuracy': accuracy,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    
    print(f"{name}:")
    print(f"  Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  CV Accuracy: {cv_scores.mean():.4f} (±{cv_scores.std():.4f})")
    print(f"\nClassification Report:")
    print(classification_report(y_test_s3, y_pred))
    print("="*60)

# Select best model
best_model_s3 = max(results_s3.items(), key=lambda x: x[1]['accuracy'])
print(f"\n🏆 Best Stage 3 Model: {best_model_s3[0]} with {best_model_s3[1]['accuracy']*100:.2f}% accuracy")
print(f"Target: 70% accuracy - {'✓ ACHIEVED!' if best_model_s3[1]['accuracy'] >= 0.70 else '❌ Not yet achieved'}")

In [ ]:
# Confusion matrix for best Stage 3 model
best_s3_name = best_model_s3[0]
best_s3_model = best_model_s3[1]['model']
y_pred_s3 = best_s3_model.predict(X_test_s3)

cm = confusion_matrix(y_test_s3, y_pred_s3)
labels = ['Dolphin', 'Minnow', 'Whale']  # Adjust based on actual classes

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=sorted(y_stage3.unique()),
            yticklabels=sorted(y_stage3.unique()))
plt.title(f'Stage 3 Confusion Matrix - {best_s3_name}')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

## 6. Complete Three-Stage Pipeline

Build the full prediction pipeline that processes brokers through all three stages.

In [ ]:
def predict_broker_classes(new_data, feature_cols, model_s1, model_s2, model_s3):
    """
    Full three-stage prediction pipeline.
    
    Parameters:
    -----------
    new_data : DataFrame
        New month's broker data with current metrics
    feature_cols : list
        List of feature column names
    model_s1, model_s2, model_s3 : trained models
        The three trained classifiers
    
    Returns:
    --------
    DataFrame with predictions for each broker
    """
    results = new_data.copy()
    X = results[feature_cols].fillna(0)
    
    # Stage 1: Predict Others vs nonOthers
    stage1_pred = model_s1.predict(X)
    results['Stage1_Prediction'] = stage1_pred
    
    # Stage 2: Confirm nonOthers
    nonothers_mask = stage1_pred == 'nonOthers'
    results['Stage2_Prediction'] = 'N/A'
    
    if nonothers_mask.sum() > 0:
        stage2_pred = model_s2.predict(X[nonothers_mask])
        results.loc[nonothers_mask, 'Stage2_Prediction'] = stage2_pred
    
    # Stage 3: Classify confirmed nonOthers into W/D/M
    confirmed_mask = (results['Stage1_Prediction'] == 'nonOthers') & \
                     (results['Stage2_Prediction'] == 'nonOthers')
    results['Final_Class'] = 'Others'
    
    if confirmed_mask.sum() > 0:
        stage3_pred = model_s3.predict(X[confirmed_mask])
        results.loc[confirmed_mask, 'Final_Class'] = stage3_pred
    
    return results[['Account Name', 'Stage1_Prediction', 'Stage2_Prediction', 'Final_Class']]

print("Pipeline function defined!")

In [ ]:
# Test the pipeline on the test set
test_predictions = predict_broker_classes(
    stage1_data,
    feature_cols,
    best_s1_model,
    best_model_s2[1]['model'],
    best_s3_model
)

print("Pipeline Test Results:")
print(f"\nTotal brokers: {len(test_predictions)}")
print(f"\nFinal Class Distribution:")
print(test_predictions['Final_Class'].value_counts())
print(f"\nSample predictions:")
print(test_predictions.head(10))

## 7. Model Summary and Next Steps

In [ ]:
# Summary of model performance
print("="*70)
print("THREE-STAGE PIPELINE PERFORMANCE SUMMARY")
print("="*70)

print(f"\n📊 STAGE 1: Others vs nonOthers")
print(f"   Best Model: {best_model_s1[0]}")
print(f"   Accuracy: {best_model_s1[1]['accuracy']*100:.2f}%")
print(f"   Target: 90.00%")
print(f"   Status: {'✅ ACHIEVED' if best_model_s1[1]['accuracy'] >= 0.90 else '⚠️  NEEDS IMPROVEMENT'}")

print(f"\n📊 STAGE 2: Confirm nonOthers")
print(f"   Best Model: {best_model_s2[0]}")
print(f"   Accuracy: {best_model_s2[1]['accuracy']*100:.2f}%")
print(f"   Target: 80.00%")
print(f"   Status: {'✅ ACHIEVED' if best_model_s2[1]['accuracy'] >= 0.80 else '⚠️  NEEDS IMPROVEMENT'}")

print(f"\n📊 STAGE 3: Whale/Dolphin/Minnow")
print(f"   Best Model: {best_model_s3[0]}")
print(f"   Accuracy: {best_model_s3[1]['accuracy']*100:.2f}%")
print(f"   Target: 70.00%")
print(f"   Status: {'✅ ACHIEVED' if best_model_s3[1]['accuracy'] >= 0.70 else '⚠️  NEEDS IMPROVEMENT'}")

print("\n" + "="*70)
print("\n📝 NEXT STEPS:")
print("   1. Save the trained models (use pickle or joblib)")
print("   2. Create a production script for monthly predictions")
print("   3. Set up monitoring for model performance drift")
print("   4. Consider hyperparameter tuning if targets not met")
print("   5. Implement feature engineering improvements if needed")
print("="*70)

In [ ]:
# Save the best models
import pickle

# Create models directory
import os
os.makedirs('../data/models', exist_ok=True)

# Save models
with open('../data/models/stage1_model.pkl', 'wb') as f:
    pickle.dump(best_s1_model, f)

with open('../data/models/stage2_model.pkl', 'wb') as f:
    pickle.dump(best_model_s2[1]['model'], f)

with open('../data/models/stage3_model.pkl', 'wb') as f:
    pickle.dump(best_s3_model, f)

# Save feature columns
with open('../data/models/feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("✅ Models saved successfully!")
print("   Location: ../data/models/")